In [6]:
# Cell 0: read MBTA CSV
from pathlib import Path
import pandas as pd

path = Path("MBTA_Commuter_Rail_Ridership_by_Trip%2C_Season%2C_Route_Line%2C_and_Stop..csv")
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")


df = pd.read_csv(path, low_memory=False)

df = df[df['season'] == 'Fall 2024']

print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns")
df.head()

Loaded 5,773 rows and 14 columns


,season,route_id,route_name,train,direction_id,day_type_id,day_type_name,stop_time,stop_id,stopsequence,average_ons,average_offs,average_load,ObjectId
9955,Fall 2024,CR-Middleborough,Middleborough/Lakeville Line,2,1,day_type_01,weekday,1/1/2024 4:50,Middleborough/Lakeville,1.0,71,0,71,9956
9958,Fall 2024,CR-Middleborough,Middleborough/Lakeville Line,2,1,day_type_01,weekday,1/1/2024 5:00,Bridgewater,2.0,31,0,102,9959
9966,Fall 2024,CR-Middleborough,Middleborough/Lakeville Line,2,1,day_type_01,weekday,1/1/2024 5:07,Campello,3.0,33,0,135,9967
9969,Fall 2024,CR-Middleborough,Middleborough/Lakeville Line,2,1,day_type_01,weekday,1/1/2024 5:11,Brockton,4.0,50,2,183,9970
9972,Fall 2024,CR-Middleborough,Middleborough/Lakeville Line,2,1,day_type_01,weekday,1/1/2024 5:14,Montello,5.0,50,1,232,9973


In [7]:
df.to_csv("MBTA_Commuter_Rail_Ridership_Fall2024.csv", index=False)

In [8]:
# aggregate boardings by route_id and day_type_name
agg_boardings = (
    df.groupby(['route_id', 'day_type_name'], as_index=False)['average_ons']
      .sum()
      .rename(columns={'average_ons': 'total_boardings'})
)

# sort for easier inspection (highest totals first)
agg_boardings = agg_boardings.sort_values(['route_id', 'total_boardings'], ascending=[True, False])


In [9]:
agg_boardings = agg_boardings[agg_boardings['day_type_name'] == 'weekday']

In [10]:
agg_boardings

,route_id,day_type_name,total_boardings
0,CR-Fairmount,weekday,4402
1,CR-Fitchburg,weekday,6388
2,CR-Franklin,weekday,10398
3,CR-Greenbush,weekday,3741
4,CR-Haverhill,weekday,3856
5,CR-Kingston,weekday,5310
6,CR-Lowell,weekday,6297
7,CR-Middleborough,weekday,6669
8,CR-Needham,weekday,6064
9,CR-Newburyport,weekday,10392


In [11]:
agg_boardings[['route_id', 'total_boardings']]
agg_boardings.to_csv("commuter_rail_route_weekday_boardings_aggregate.csv", index=False)

In [12]:
total_boardings_sum = agg_boardings['total_boardings'].sum()
print(f"Sum of total_boardings: {total_boardings_sum:,.1f}")

Sum of total_boardings: 97,537.0


In [13]:
# Scale total_boardings so that their sum matches 111,755
target_sum = 111_755

scale_factor = target_sum / total_boardings_sum

agg_boardings_scaled = agg_boardings.copy()
agg_boardings_scaled['total_boardings'] = agg_boardings_scaled['total_boardings'] * scale_factor

print(f"Scaled sum: {agg_boardings_scaled['total_boardings'].sum():,.1f}")
agg_boardings_scaled.head()

Scaled sum: 111,755.0


,route_id,day_type_name,total_boardings
0,CR-Fairmount,weekday,5043.680962
1,CR-Fitchburg,weekday,7319.180824
2,CR-Franklin,weekday,11913.719819
3,CR-Greenbush,weekday,4286.326779
4,CR-Haverhill,weekday,4418.090366


In [14]:
agg_boardings_scaled.to_csv("agg_commuter_rail_boardings_scaled.csv", index=False)
agg_boardings_scaled


,route_id,day_type_name,total_boardings
0,CR-Fairmount,weekday,5043.680962
1,CR-Fitchburg,weekday,7319.180824
2,CR-Franklin,weekday,11913.719819
3,CR-Greenbush,weekday,4286.326779
4,CR-Haverhill,weekday,4418.090366
5,CR-Kingston,weekday,6084.040415
6,CR-Lowell,weekday,7214.915724
7,CR-Middleborough,weekday,7641.142284
8,CR-Needham,weekday,6947.951239
9,CR-Newburyport,weekday,11906.845197


In [15]:
# aggregate statistics on agg_boardings.total_boardings (weekday)
tb = agg_boardings['total_boardings']

agg_stats = pd.Series({
    'count': tb.count(),
    'sum': tb.sum(),
    'mean': tb.mean(),
    'median': tb.median(),
    'std': tb.std(),
    'min': tb.min(),
    'max': tb.max()
})
print(agg_stats)

# re-aggregate by route_id (defensive) and sort
per_route = agg_boardings.groupby('route_id', as_index=False)['total_boardings'].sum()
per_route = per_route.sort_values('total_boardings', ascending=False)

# top / bottom routes
print("\nTop 10 routes by weekday total_boardings:")
print(per_route.head(10))

print("\nBottom 10 routes by weekday total_boardings:")
print(per_route.tail(10))

# add percent and cumulative share, save results
per_route['pct_share'] = per_route['total_boardings'] / per_route['total_boardings'].sum()
per_route['cum_share'] = per_route['pct_share'].cumsum()
per_route.to_csv("weekday_boardings_per_route_with_shares.csv", index=False)

# quick bar plot of top 20 routes (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    per_route.set_index('route_id')['total_boardings'].head(20).plot(
        kind='bar', figsize=(10,4), title='Top 20 routes — weekday total boardings'
    )
    plt.ylabel('total_boardings')
    plt.tight_layout()
except Exception:
    pass

count        12.000000
sum       97537.000000
mean       8128.083333
median     6342.500000
std        4754.907349
min        3741.000000
max       19078.000000
dtype: float64

Top 10 routes by weekday total_boardings:
            route_id  total_boardings
10     CR-Providence            19078
11      CR-Worcester            14942
2        CR-Franklin            10398
9     CR-Newburyport            10392
7   CR-Middleborough             6669
1       CR-Fitchburg             6388
6          CR-Lowell             6297
8         CR-Needham             6064
5        CR-Kingston             5310
0       CR-Fairmount             4402

Bottom 10 routes by weekday total_boardings:
           route_id  total_boardings
2       CR-Franklin            10398
9    CR-Newburyport            10392
7  CR-Middleborough             6669
1      CR-Fitchburg             6388
6         CR-Lowell             6297
8        CR-Needham             6064
5       CR-Kingston             5310
0      CR-Fairmount  